# Course 1 · Week 3 — Hands-on: Classification with Logistic Regression

In Week 1 you learned to predict a **number** (regression). In Week 2 you scaled it up to many features. Now: predict a **category**.

By the end you'll have:

1. Implemented the sigmoid function — the smooth on/off switch
2. Implemented logistic regression's prediction, cost (log-loss), and gradients
3. Trained a classifier on a 2D dataset
4. Plotted the **decision boundary** that separates the two classes
5. (Stretch) Added regularization and watched it tame an overfitting model

Estimated time: **45–60 minutes.**


## Setup

Two 2D point clusters: class 0 around the bottom-left, class 1 around the top-right. They overlap a bit on purpose — perfect classification would mean the data isn't realistic. Real data always has overlap; the goal is to find the line that does the best you can.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(3)
m = 60   # 60 examples, 30 per class

# Two clusters in 2D — class 0 around (-1, -1), class 1 around (+1, +1), with overlap
X0 = np.random.randn(m // 2, 2) * 0.7 + np.array([-1.0, -1.0])
X1 = np.random.randn(m // 2, 2) * 0.7 + np.array([+1.0, +1.0])
X = np.vstack([X0, X1])
y = np.concatenate([np.zeros(m // 2), np.ones(m // 2)])  # labels: 0 or 1

print(f"X.shape = {X.shape},  y.shape = {y.shape}")
print(f"class 0 examples: {int((y==0).sum())},  class 1 examples: {int((y==1).sum())}")


You should see two blobs of points. The red dots are class 0, the green triangles are class 1. They mostly stay in their corners but a few wander into enemy territory.


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color="#ef4444", s=70, label="class 0", alpha=0.8)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color="#10b981", s=70, label="class 1", alpha=0.8, marker="^")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$")
plt.title("Two slightly overlapping classes")
plt.legend(); plt.grid(alpha=0.3); plt.axhline(0, color="black", lw=0.5); plt.axvline(0, color="black", lw=0.5)
plt.show()


## Quick recap

Linear regression's output is any real number — perfect for predicting price. But for a **yes/no** question like "is this email spam?" or "is this tumor malignant?", you need the output to live in $[0, 1]$ — interpretable as a probability.

The fix: take a linear model's output and squash it through the **sigmoid** function:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

The sigmoid is a soft on/off switch: very negative inputs → near 0, very positive → near 1, zero → exactly 0.5.

Logistic regression's prediction is then:

$$p = \sigma(\vec w \cdot \vec x + b)$$

The "logistic" in the name has nothing to do with logistics. It comes from the *logistic function* — another name for the sigmoid.


## Exercise 1 — sigmoid

Implement `sigmoid(z)` so it works for both scalars and numpy arrays.

Math: $\sigma(z) = \frac{1}{1 + e^{-z}}$. Numpy provides `np.exp`.


In [ ]:
def sigmoid(z):
    """Squash a real number (or array of reals) into (0, 1).

    Math: sigmoid(z) = 1 / (1 + e^{-z})

    Args:
        z: scalar or numpy array.

    Returns:
        Same shape as z, with all values in (0, 1).
    """
    # TODO: implement using np.exp
    return 0.5  # placeholder


# Sanity check
print(f"sigmoid(0)  = {sigmoid(0.0):.4f}   (expected 0.5000)")
print(f"sigmoid(2)  = {sigmoid(2.0):.4f}   (expected 0.8808)")
print(f"sigmoid(-2) = {sigmoid(-2.0):.4f}   (expected 0.1192)")

assert abs(sigmoid(0.0) - 0.5) < 1e-9
assert abs(sigmoid(2.0) - 0.8807970779778823) < 1e-9
assert abs(sigmoid(-2.0) - 0.11920292202211755) < 1e-9
print("✓ sigmoid() works")


## Exercise 2 — predict

Combine the linear model from Week 2 with the sigmoid:

$$p = \sigma(\vec w \cdot \vec x + b)$$

In numpy, that's just `sigmoid(X @ w + b)`.


In [ ]:
def predict(X, w, b):
    """Logistic-regression probability that each example is class 1.

    Math: P(y=1 | x) = sigmoid(w·x + b)
    """
    # TODO: combine your linear model from week 2 with the sigmoid function you just wrote
    return np.zeros(len(X))


# Quick check — at (w=0, b=0), every probability should be 0.5
p = predict(X, np.zeros(2), 0.0)
print(f"first 5 probs at zeros: {np.round(p[:5], 4)}")
assert np.allclose(p, 0.5)
print("✓ predict() works")


## Exercise 3 — cost (log-loss)

Implement binary cross-entropy:

$$J(w, b) = -\frac{1}{m} \sum_{i=1}^{m} \bigl[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \bigr]$$

We `np.clip(p, eps, 1 - eps)` to keep `log(0)` from blowing up if a prediction lands exactly at 0 or 1.

Anchored value: at zeros, every `p = 0.5`, so each per-example loss is $-\log(0.5) = \log 2 \approx 0.6931$. Average over all examples is the same.


In [ ]:
def cost(X, y, w, b):
    """Binary cross-entropy / log-loss.

    Math: J = -(1/m) * sum( y * log(p) + (1-y) * log(1-p) )

    where p = sigmoid(w·x + b) is the predicted probability of class 1.
    """
    p = predict(X, w, b)
    eps = 1e-15
    p = np.clip(p, eps, 1 - eps)   # avoid log(0)
    # TODO: implement the log-loss using y, p, np.log, np.mean
    return 0.0


J0 = cost(X, y, np.zeros(2), 0.0)
print(f"cost at zeros = {J0:.6f}   (expected ≈ 0.6931, which is ln(2))")
assert abs(J0 - 0.6931471805599453) < 1e-6
print("✓ cost() works")


## Exercise 4 — gradients

Same form as linear regression. `dw` and `db` are averages of `(p - y) * features` and `(p - y)` respectively.


In [ ]:
def gradients(X, y, w, b):
    """Gradients of the log-loss.

    Surprise: same form as linear regression!
      dJ/dw_j = (1/m) * sum( (p_i - y_i) * X[i, j] )
      dJ/db   = (1/m) * sum(  p_i - y_i )

    where p_i = sigmoid(w·x_i + b).
    """
    # TODO: implement.
    return np.zeros(X.shape[1]), 0.0


dw0, db0 = gradients(X, y, np.zeros(2), 0.0)
print(f"dw at zeros = {np.round(dw0, 4)}   (expected ≈ [-0.4815, -0.6296])")
print(f"db at zeros = {db0:.4f}        (expected ≈ 0.0000)")
assert abs(dw0[0] - (-0.4815)) < 0.001
assert abs(dw0[1] - (-0.6296)) < 0.001
assert abs(db0) < 0.001
print("✓ gradients() works")


## Exercise 5 — gradient descent

Same shape as Weeks 1 and 2. Run for 5000 iterations with `alpha=0.5`. Then compute training accuracy: predict, threshold at 0.5, count matches.


In [ ]:
def gradient_descent(X, y, w0, b0, alpha, n_iters):
    """Run gradient descent. Returns (w, b, history)."""
    w = w0.copy()
    b = b0
    history = []
    for _ in range(n_iters):
        # TODO: simultaneous update + record cost
        pass
    return w, b, history


# Train
w_final, b_final, hist = gradient_descent(X, y, np.zeros(2), 0.0, alpha=0.5, n_iters=5000)
print(f"w = {np.round(w_final, 3)}")
print(f"b = {b_final:.3f}")
print(f"final cost = {hist[-1] if hist else 'N/A'}")

# Accuracy on training set
p_final = predict(X, w_final, b_final)
y_pred = (p_final >= 0.5).astype(int)
acc = (y_pred == y).mean()
print(f"training accuracy = {acc:.4f}")


## Visualizing the decision boundary

The boundary is where the model is exactly 50/50: `w · x + b = 0`. In 2D:

$$w_1 x_1 + w_2 x_2 + b = 0 \quad \Rightarrow \quad x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$$

That's a straight line. Plot it on top of the scatter and you can see the separator the model learned.


In [ ]:
# Plot the decision boundary.
# The boundary is where w·x + b = 0, i.e. where p = 0.5.
# In 2D: w0*x0 + w1*x1 + b = 0  →  x1 = -(w0/w1)*x0 - b/w1

xs = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
ys_boundary = -(w_final[0] / w_final[1]) * xs - b_final / w_final[1]

plt.figure(figsize=(7, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color="#ef4444", s=70, label="class 0", alpha=0.8)
plt.scatter(X[y == 1, 0], X[y == 1, 1], color="#10b981", s=70, label="class 1", alpha=0.8, marker="^")
plt.plot(xs, ys_boundary, color="#4338ca", lw=2.5, label="decision boundary")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.legend(); plt.grid(alpha=0.3)
plt.title(f"Logistic regression — accuracy {acc:.1%}")
plt.show()

# Also plot the cost over iterations
plt.figure(figsize=(7, 3.5))
plt.plot(hist, color="#4338ca")
plt.xlabel("iteration"); plt.ylabel("cost"); plt.yscale("log"); plt.grid(alpha=0.3)
plt.title("Cost decreasing during training")
plt.show()


## ⭐ Stretch — regularization

Logistic regression with only 2 raw features (`x_1`, `x_2`) underfits — it can only draw a straight line, but the data might really need a curve. So we'll *invent* polynomial features: `x_1²`, `x_1 x_2`, `x_1³`, etc. Now the model can draw curves.

But it can also draw too-wiggly curves. With 9 polynomial features and only 60 examples, the model can memorize noise — perfect training accuracy, terrible generalization.

The fix is **regularization**: add a penalty for big weights to the cost. Cost becomes:

$$J_\text{reg} = J + \frac{\lambda}{2m} \sum_{j} w_j^2$$

The `lambda` knob controls how much you penalize. Lambda = 0 is "do whatever, memorize away." Large lambda is "stay simple, even if you have to be wrong on training."

Compare lambda = 0, 0.1, 1.0, 10.0 and look at training accuracy AND `||w||` (the magnitude of weights).


In [ ]:
# STRETCH: Regularization
# Add polynomial features (up to cubic) to MAKE the model overfittable, then
# compare lambda=0, 0.1, 1.0, 10.0.

X_poly = np.column_stack([
    X[:, 0], X[:, 1],
    X[:, 0]**2, X[:, 1]**2, X[:, 0]*X[:, 1],
    X[:, 0]**3, X[:, 1]**3, X[:, 0]**2 * X[:, 1], X[:, 0] * X[:, 1]**2,
])

# Scale (z-score)
mu = X_poly.mean(axis=0); sigma = X_poly.std(axis=0)
X_poly_s = (X_poly - mu) / sigma

# TODO: write a regularized version of cost and gradients:
#   cost_reg = cost + (lambda / 2m) * sum(w^2)
#   dw_reg   = dw   + (lambda / m)  * w     # b is NOT regularized
# Then for each lambda in [0, 0.1, 1.0, 10.0], train and report accuracy and ||w||.

# Hints:
#   def gradients_reg(X, y, w, b, lam): ...
#   def gradient_descent_reg(X, y, w0, b0, alpha, n_iters, lam): ...


## Wrap-up

You just built a binary classifier from scratch with regularization. Three weeks of Course 1 in three notebooks. Recap of the universal pattern:

1. Define a model: `predict(X, parameters)`
2. Define a cost: `cost(X, y, parameters)`
3. Compute gradients
4. Run gradient descent

Linear regression, multivariate regression, polynomial regression, logistic regression — all the same skeleton. Only the model and cost change.

In Course 2 you'll see what happens when "the model" becomes a neural network. The skeleton stays exactly the same.

🎉 You've completed Course 1.
